In [13]:
from pathlib import Path
from collections import defaultdict
import re
import warnings

import numpy as np
import pandas as pd
import joblib

from scapy.all import rdpcap, IP, TCP, UDP, ICMP, Raw
from scapy.layers.dns import DNS, DNSQR, DNSRR

warnings.filterwarnings("ignore")

PCAP_PATH = Path("Dataset-2.pcap")

MODEL_PATH = Path("artifacts/xgb_nfv2_model.joblib")
ENCODER_PATH = Path("artifacts/label_encoder.joblib")
FEATURE_PATH = Path("artifacts/feature_names.joblib")

OUTPUT_PATH = Path("deneme_nfv2_features.csv")

assert PCAP_PATH.exists(), f"PCAP bulunamadi: {PCAP_PATH}"
assert MODEL_PATH.exists(), f"Model bulunamadi: {MODEL_PATH}"
assert ENCODER_PATH.exists(), f"Encoder bulunamadi: {ENCODER_PATH}"
assert FEATURE_PATH.exists(), f"Feature listesi bulunamadi: {FEATURE_PATH}"

print("Yollar hazir.")

Yollar hazir.


In [14]:
packets = rdpcap(str(PCAP_PATH))

model = joblib.load(MODEL_PATH)
label_encoder = joblib.load(ENCODER_PATH)
feature_names = joblib.load(FEATURE_PATH)

print("Okunan paket:", len(packets))
print("Model feature sayisi:", model.n_features_in_)
print("Kayitli feature sayisi:", len(feature_names))
print("Sinif sayisi:", len(label_encoder.classes_))

assert model.n_features_in_ == 39
assert len(feature_names) == 39

Okunan paket: 517
Model feature sayisi: 39
Kayitli feature sayisi: 39
Sinif sayisi: 16


In [16]:
FTP_REPLY_PATTERN = re.compile(rb"^(\d{3})[\s-]")


def get_transport_info(pkt):
    if TCP in pkt:
        return (
            int(pkt[TCP].sport),
            int(pkt[TCP].dport),
            6
        )

    if UDP in pkt:
        return (
            int(pkt[UDP].sport),
            int(pkt[UDP].dport),
            17
        )

    if ICMP in pkt:
        return 0, 0, 1

    if IP in pkt:
        return 0, 0, int(pkt[IP].proto)

    return 0, 0, 0


def make_forward_key(src, dst, sport, dport, proto):
    return (
        src,
        dst,
        int(sport),
        int(dport),
        int(proto)
    )


def make_reverse_key(src, dst, sport, dport, proto):
    return (
        dst,
        src,
        int(dport),
        int(sport),
        int(proto)
    )


def directional_duration_ms(first_ts, last_ts):
    if first_ts is None or last_ts is None:
        return 0.0

    return max(
        0.0,
        (float(last_ts) - float(first_ts)) * 1000.0
    )


def calculate_second_bytes(byte_count, first_ts, last_ts):
    if byte_count <= 0:
        return 0.0

    duration_ms = directional_duration_ms(
        first_ts,
        last_ts
    )

    # Tek paketli yönlerde 1 ms taban pencere
    effective_seconds = max(
        duration_ms / 1000.0,
        0.001
    )

    return float(byte_count) / effective_seconds


def update_packet_size_bin(flow, packet_size):
    if packet_size <= 128:
        flow["NUM_PKTS_UP_TO_128_BYTES"] += 1

    elif packet_size <= 256:
        flow["NUM_PKTS_128_TO_256_BYTES"] += 1

    elif packet_size <= 512:
        flow["NUM_PKTS_256_TO_512_BYTES"] += 1

    elif packet_size <= 1024:
        flow["NUM_PKTS_512_TO_1024_BYTES"] += 1

    elif packet_size <= 1514:
        flow["NUM_PKTS_1024_TO_1514_BYTES"] += 1


def tcp_sequence_range(pkt):
    if TCP not in pkt:
        return None

    tcp = pkt[TCP]

    payload_length = len(bytes(tcp.payload))
    consumed_sequence = payload_length
    flags = int(tcp.flags)

    # SYN
    if flags & 0x02:
        consumed_sequence += 1

    # FIN
    if flags & 0x01:
        consumed_sequence += 1

    if consumed_sequence <= 0:
        return None

    start = int(tcp.seq)
    end = start + consumed_sequence

    return start, end, payload_length


def ranges_overlap(start_a, end_a, start_b, end_b):
    return start_a < end_b and start_b < end_a


def first_dns_answer_ttl(dns):
    if int(dns.ancount or 0) <= 0:
        return 0

    ttls = []

    current = dns.an

    for _ in range(int(dns.ancount or 0)):
        if current is None:
            break

        try:
            if hasattr(current, "ttl"):
                ttls.append(int(current.ttl))
        except (TypeError, ValueError):
            pass

        try:
            current = current.payload
        except AttributeError:
            break

    return min(ttls) if ttls else 0


def detect_l7_protocol(pkt, source_port, destination_port):
    # Mevcut extractor mapping'i korunuyor.
    # Eğitim verisindeki L7_PROTO ID sözlüğü ayrıca doğrulanmalı.

    if DNS in pkt:
        return 1

    if source_port == 21 or destination_port == 21:
        return 2

    if (
        source_port in {443, 8443}
        or destination_port in {443, 8443}
    ):
        return 3

    return 0

In [17]:
def create_flow(
    source_ip,
    destination_ip,
    source_port,
    destination_port,
    protocol,
    timestamp
):
    return {
        "source_ip": source_ip,
        "destination_ip": destination_ip,
        "source_port": int(source_port),
        "destination_port": int(destination_port),
        "protocol": int(protocol),

        "first_ts": float(timestamp),
        "last_ts": float(timestamp),

        "first_in_ts": None,
        "last_in_ts": None,

        "first_out_ts": None,
        "last_out_ts": None,

        "IN_BYTES": 0,
        "IN_PKTS": 0,

        "OUT_BYTES": 0,
        "OUT_PKTS": 0,

        "TCP_FLAGS": 0,
        "CLIENT_TCP_FLAGS": 0,
        "SERVER_TCP_FLAGS": 0,

        "MIN_TTL": None,
        "MAX_TTL": 0,

        "LONGEST_FLOW_PKT": 0,
        "SHORTEST_FLOW_PKT": None,

        "MIN_IP_PKT_LEN": None,
        "MAX_IP_PKT_LEN": 0,

        "NUM_PKTS_UP_TO_128_BYTES": 0,
        "NUM_PKTS_128_TO_256_BYTES": 0,
        "NUM_PKTS_256_TO_512_BYTES": 0,
        "NUM_PKTS_512_TO_1024_BYTES": 0,
        "NUM_PKTS_1024_TO_1514_BYTES": 0,

        "TCP_WIN_MAX_IN": 0,
        "TCP_WIN_MAX_OUT": 0,

        "RETRANSMITTED_IN_BYTES": 0,
        "RETRANSMITTED_IN_PKTS": 0,

        "RETRANSMITTED_OUT_BYTES": 0,
        "RETRANSMITTED_OUT_PKTS": 0,

        "seen_tcp_ranges_in": [],
        "seen_tcp_ranges_out": [],

        "ICMP_TYPE": 0,
        "ICMP_IPV4_TYPE": 0,

        "DNS_QUERY_ID": 0,
        "DNS_QUERY_TYPE": 0,
        "DNS_TTL_ANSWER": 0,

        "FTP_COMMAND_RET_CODE": 0,

        "L7_PROTO": 0
    }

In [18]:
def update_directional_statistics(
    flow,
    pkt,
    direction
):
    timestamp = float(pkt.time)

    # NF/IP byte sayımı için Ethernet frame değil,
    # IP paket uzunluğunu kullanıyoruz.
    ip_packet_length = int(
        pkt[IP].len or len(pkt[IP])
    )

    flow["last_ts"] = max(
        flow["last_ts"],
        timestamp
    )

    if direction == "in":
        flow["IN_BYTES"] += ip_packet_length
        flow["IN_PKTS"] += 1

        if flow["first_in_ts"] is None:
            flow["first_in_ts"] = timestamp

        flow["last_in_ts"] = timestamp

    else:
        flow["OUT_BYTES"] += ip_packet_length
        flow["OUT_PKTS"] += 1

        if flow["first_out_ts"] is None:
            flow["first_out_ts"] = timestamp

        flow["last_out_ts"] = timestamp


def update_length_statistics(flow, pkt):
    frame_length = len(pkt)

    ip_packet_length = int(
        pkt[IP].len or len(pkt[IP])
    )

    flow["LONGEST_FLOW_PKT"] = max(
        flow["LONGEST_FLOW_PKT"],
        frame_length
    )

    if flow["SHORTEST_FLOW_PKT"] is None:
        flow["SHORTEST_FLOW_PKT"] = frame_length
    else:
        flow["SHORTEST_FLOW_PKT"] = min(
            flow["SHORTEST_FLOW_PKT"],
            frame_length
        )

    flow["MAX_IP_PKT_LEN"] = max(
        flow["MAX_IP_PKT_LEN"],
        ip_packet_length
    )

    if flow["MIN_IP_PKT_LEN"] is None:
        flow["MIN_IP_PKT_LEN"] = ip_packet_length
    else:
        flow["MIN_IP_PKT_LEN"] = min(
            flow["MIN_IP_PKT_LEN"],
            ip_packet_length
        )

    ttl = int(pkt[IP].ttl)

    if flow["MIN_TTL"] is None:
        flow["MIN_TTL"] = ttl
    else:
        flow["MIN_TTL"] = min(
            flow["MIN_TTL"],
            ttl
        )

    flow["MAX_TTL"] = max(
        flow["MAX_TTL"],
        ttl
    )

    update_packet_size_bin(
        flow,
        frame_length
    )


def update_tcp_statistics(
    flow,
    pkt,
    direction
):
    if TCP not in pkt:
        return

    flags = int(pkt[TCP].flags)
    window = int(pkt[TCP].window or 0)

    flow["TCP_FLAGS"] |= flags

    if direction == "in":
        flow["CLIENT_TCP_FLAGS"] |= flags

        flow["TCP_WIN_MAX_IN"] = max(
            flow["TCP_WIN_MAX_IN"],
            window
        )

    else:
        flow["SERVER_TCP_FLAGS"] |= flags

        flow["TCP_WIN_MAX_OUT"] = max(
            flow["TCP_WIN_MAX_OUT"],
            window
        )


def update_retransmission_statistics(
    flow,
    pkt,
    direction
):
    sequence_info = tcp_sequence_range(pkt)

    if sequence_info is None:
        return

    start, end, payload_length = sequence_info

    ranges_key = (
        "seen_tcp_ranges_in"
        if direction == "in"
        else "seen_tcp_ranges_out"
    )

    is_retransmission = any(
        ranges_overlap(
            start,
            end,
            old_start,
            old_end
        )
        for old_start, old_end
        in flow[ranges_key]
    )

    if is_retransmission:
        if direction == "in":
            flow["RETRANSMITTED_IN_PKTS"] += 1
            flow["RETRANSMITTED_IN_BYTES"] += payload_length

        else:
            flow["RETRANSMITTED_OUT_PKTS"] += 1
            flow["RETRANSMITTED_OUT_BYTES"] += payload_length

    else:
        flow[ranges_key].append(
            (start, end)
        )


def update_dns_statistics(flow, pkt):
    if DNS not in pkt:
        return

    dns = pkt[DNS]

    flow["L7_PROTO"] = 1
    flow["DNS_QUERY_ID"] = int(dns.id or 0)

    if dns.qd is not None:
        try:
            question = dns.qd

            if isinstance(question, list):
                question = question[0]

            flow["DNS_QUERY_TYPE"] = int(
                question.qtype or 0
            )

        except (
            AttributeError,
            TypeError,
            ValueError,
            IndexError
        ):
            pass

    if int(dns.qr or 0) == 1:
        answer_ttl = first_dns_answer_ttl(dns)

        if answer_ttl > 0:
            flow["DNS_TTL_ANSWER"] = answer_ttl


def update_icmp_statistics(flow, pkt):
    if ICMP not in pkt:
        return

    icmp_type = int(
        pkt[ICMP].type or 0
    )

    flow["ICMP_TYPE"] = icmp_type
    flow["ICMP_IPV4_TYPE"] = icmp_type


def update_ftp_statistics(flow, pkt):
    if TCP not in pkt or Raw not in pkt:
        return

    source_port = int(pkt[TCP].sport)
    destination_port = int(pkt[TCP].dport)

    if (
        source_port != 21
        and destination_port != 21
    ):
        return

    flow["L7_PROTO"] = 2

    payload = bytes(pkt[Raw].load)

    match = FTP_REPLY_PATTERN.match(payload)

    if match:
        flow["FTP_COMMAND_RET_CODE"] = int(
            match.group(1)
        )

In [19]:
flows = {}

non_ipv4_count = 0

for pkt in packets:
    if IP not in pkt:
        non_ipv4_count += 1
        continue

    source_ip = pkt[IP].src
    destination_ip = pkt[IP].dst

    source_port, destination_port, protocol = (
        get_transport_info(pkt)
    )

    forward_key = make_forward_key(
        source_ip,
        destination_ip,
        source_port,
        destination_port,
        protocol
    )

    reverse_key = make_reverse_key(
        source_ip,
        destination_ip,
        source_port,
        destination_port,
        protocol
    )

    if forward_key in flows:
        flow_key = forward_key
        direction = "in"

    elif reverse_key in flows:
        flow_key = reverse_key
        direction = "out"

    else:
        # İlk görülen paket initiator/client yönüdür.
        flow_key = forward_key
        direction = "in"

        flows[flow_key] = create_flow(
            source_ip=source_ip,
            destination_ip=destination_ip,
            source_port=source_port,
            destination_port=destination_port,
            protocol=protocol,
            timestamp=float(pkt.time)
        )

    flow = flows[flow_key]

    update_directional_statistics(
        flow,
        pkt,
        direction
    )

    update_length_statistics(
        flow,
        pkt
    )

    update_tcp_statistics(
        flow,
        pkt,
        direction
    )

    update_retransmission_statistics(
        flow,
        pkt,
        direction
    )

    update_dns_statistics(
        flow,
        pkt
    )

    update_icmp_statistics(
        flow,
        pkt
    )

    update_ftp_statistics(
        flow,
        pkt
    )

    detected_l7 = detect_l7_protocol(
        pkt,
        source_port,
        destination_port
    )

    if flow["L7_PROTO"] == 0:
        flow["L7_PROTO"] = detected_l7

print("Okunan paket:", len(packets))
print("Atlanan IPv4 olmayan paket:", non_ipv4_count)
print("Olusturulan bidirectional flow:", len(flows))


Okunan paket: 517
Atlanan IPv4 olmayan paket: 24
Olusturulan bidirectional flow: 64


In [20]:
def finalize_flow(flow):
    flow_duration_ms = max(
        0.0,
        (
            flow["last_ts"]
            - flow["first_ts"]
        ) * 1000.0
    )

    duration_in_ms = directional_duration_ms(
        flow["first_in_ts"],
        flow["last_in_ts"]
    )

    duration_out_ms = directional_duration_ms(
        flow["first_out_ts"],
        flow["last_out_ts"]
    )

    src_second_bytes = calculate_second_bytes(
        flow["IN_BYTES"],
        flow["first_in_ts"],
        flow["last_in_ts"]
    )

    dst_second_bytes = calculate_second_bytes(
        flow["OUT_BYTES"],
        flow["first_out_ts"],
        flow["last_out_ts"]
    )

    src_avg_throughput = (
        src_second_bytes * 8.0
    )

    dst_avg_throughput = (
        dst_second_bytes * 8.0
    )

    return {
        "PROTOCOL": flow["protocol"],
        "L7_PROTO": flow["L7_PROTO"],

        "IN_BYTES": flow["IN_BYTES"],
        "IN_PKTS": flow["IN_PKTS"],

        "OUT_BYTES": flow["OUT_BYTES"],
        "OUT_PKTS": flow["OUT_PKTS"],

        "TCP_FLAGS": flow["TCP_FLAGS"],
        "CLIENT_TCP_FLAGS":
            flow["CLIENT_TCP_FLAGS"],
        "SERVER_TCP_FLAGS":
            flow["SERVER_TCP_FLAGS"],

        "FLOW_DURATION_MILLISECONDS":
            flow_duration_ms,

        "DURATION_IN": duration_in_ms,
        "DURATION_OUT": duration_out_ms,

        "MIN_TTL":
            flow["MIN_TTL"]
            if flow["MIN_TTL"] is not None
            else 0,

        "MAX_TTL": flow["MAX_TTL"],

        "LONGEST_FLOW_PKT":
            flow["LONGEST_FLOW_PKT"],

        "SHORTEST_FLOW_PKT":
            flow["SHORTEST_FLOW_PKT"]
            if flow["SHORTEST_FLOW_PKT"] is not None
            else 0,

        "MIN_IP_PKT_LEN":
            flow["MIN_IP_PKT_LEN"]
            if flow["MIN_IP_PKT_LEN"] is not None
            else 0,

        "MAX_IP_PKT_LEN":
            flow["MAX_IP_PKT_LEN"],

        "SRC_TO_DST_SECOND_BYTES":
            src_second_bytes,

        "DST_TO_SRC_SECOND_BYTES":
            dst_second_bytes,

        "RETRANSMITTED_IN_BYTES":
            flow["RETRANSMITTED_IN_BYTES"],

        "RETRANSMITTED_IN_PKTS":
            flow["RETRANSMITTED_IN_PKTS"],

        "RETRANSMITTED_OUT_BYTES":
            flow["RETRANSMITTED_OUT_BYTES"],

        "RETRANSMITTED_OUT_PKTS":
            flow["RETRANSMITTED_OUT_PKTS"],

        "SRC_TO_DST_AVG_THROUGHPUT":
            src_avg_throughput,

        "DST_TO_SRC_AVG_THROUGHPUT":
            dst_avg_throughput,

        "NUM_PKTS_UP_TO_128_BYTES":
            flow["NUM_PKTS_UP_TO_128_BYTES"],

        "NUM_PKTS_128_TO_256_BYTES":
            flow["NUM_PKTS_128_TO_256_BYTES"],

        "NUM_PKTS_256_TO_512_BYTES":
            flow["NUM_PKTS_256_TO_512_BYTES"],

        "NUM_PKTS_512_TO_1024_BYTES":
            flow["NUM_PKTS_512_TO_1024_BYTES"],

        "NUM_PKTS_1024_TO_1514_BYTES":
            flow["NUM_PKTS_1024_TO_1514_BYTES"],

        "TCP_WIN_MAX_IN":
            flow["TCP_WIN_MAX_IN"],

        "TCP_WIN_MAX_OUT":
            flow["TCP_WIN_MAX_OUT"],

        "ICMP_TYPE":
            flow["ICMP_TYPE"],

        "ICMP_IPV4_TYPE":
            flow["ICMP_IPV4_TYPE"],

        "DNS_QUERY_ID":
            flow["DNS_QUERY_ID"],

        "DNS_QUERY_TYPE":
            flow["DNS_QUERY_TYPE"],

        "DNS_TTL_ANSWER":
            flow["DNS_TTL_ANSWER"],

        "FTP_COMMAND_RET_CODE":
            flow["FTP_COMMAND_RET_CODE"]
    }


feature_rows = [
    finalize_flow(flow)
    for flow in flows.values()
]

features = pd.DataFrame(feature_rows)

print("Ham shape:", features.shape)
print(features.head())

Ham shape: (64, 39)
   PROTOCOL  L7_PROTO  IN_BYTES  IN_PKTS  OUT_BYTES  OUT_PKTS  TCP_FLAGS  \
0         6         0       435        6        395         5         27   
1         6         0        44        1         40         1         24   
2         6         0       435        6        395         5         27   
3         6         0       435        6        395         5         27   
4         6         3       933       11        305         5         24   

   CLIENT_TCP_FLAGS  SERVER_TCP_FLAGS  FLOW_DURATION_MILLISECONDS  ...  \
0                27                27                   84.347963  ...   
1                24                16                   41.357994  ...   
2                27                27                   90.003014  ...   
3                27                27                   30.715942  ...   
4                24                24                 8083.656073  ...   

   NUM_PKTS_512_TO_1024_BYTES  NUM_PKTS_1024_TO_1514_BYTES  TCP_WIN_MAX_IN  \


In [21]:
missing_columns = [
    column
    for column in feature_names
    if column not in features.columns
]

extra_columns = [
    column
    for column in features.columns
    if column not in feature_names
]

print("Eksik kolonlar:", missing_columns)
print("Fazla kolonlar:", extra_columns)

assert not missing_columns, (
    f"Eksik kolon bulundu: {missing_columns}"
)

features = features[
    feature_names
].copy()

features = features.apply(
    pd.to_numeric,
    errors="coerce"
)

features = features.replace(
    [np.inf, -np.inf],
    np.nan
)

features = features.fillna(0.0)

FLOAT32_SAFE_LIMIT = (
    np.finfo(np.float32).max / 10.0
)

features = features.clip(
    lower=-FLOAT32_SAFE_LIMIT,
    upper=FLOAT32_SAFE_LIMIT
)

features = features.astype(
    np.float32
)

assert features.shape[1] == 39
assert features.isna().sum().sum() == 0
assert np.isfinite(
    features.to_numpy()
).all()

features.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nFeature CSV kaydedildi:")
print(OUTPUT_PATH.resolve())

print("\nShape:")
print(features.shape)

print("\nNaN sayisi:")
print(int(features.isna().sum().sum()))

print("\nTum degerler finite mi?")
print(
    np.isfinite(
        features.to_numpy()
    ).all()
)

Eksik kolonlar: []
Fazla kolonlar: []

Feature CSV kaydedildi:
C:\Users\egemen.keles\PycharmProjects\mitre_mapper\deneme_nfv2_features.csv

Shape:
(64, 39)

NaN sayisi:
0

Tum degerler finite mi?
True


In [22]:
probabilities = model.predict_proba(
    features
)

top3_indices = np.argsort(
    probabilities,
    axis=1
)[:, -3:][:, ::-1]

report_rows = []

flow_keys = list(flows.keys())

for flow_index in range(len(features)):
    flow_key = flow_keys[flow_index]

    row = {
        "flow_index": flow_index,

        "src_ip": flow_key[0],
        "dst_ip": flow_key[1],

        "src_port": flow_key[2],
        "dst_port": flow_key[3],

        "protocol": flow_key[4]
    }

    for rank, class_index in enumerate(
        top3_indices[flow_index],
        start=1
    ):
        label = label_encoder.inverse_transform(
            np.array(
                [class_index],
                dtype=np.int32
            )
        )[0]

        probability = probabilities[
            flow_index,
            class_index
        ]

        row[f"top_{rank}_label"] = label
        row[f"top_{rank}_probability"] = float(
            probability
        )

    report_rows.append(row)

prediction_report = pd.DataFrame(
    report_rows
)

print(
    prediction_report.to_string(
        index=False
    )
)

 flow_index        src_ip          dst_ip  src_port  dst_port  protocol    top_1_label  top_1_probability    top_2_label  top_2_probability    top_3_label  top_3_probability
          0    10.50.0.20      10.34.2.36     52587      7680         6         Benign           0.895254    Brute Force           0.045400 Reconnaissance           0.020677
          1    10.5.29.57      10.34.2.36      7680     53655         6         Benign           0.968271 Reconnaissance           0.022907            DoS           0.002935
          2     10.2.0.62      10.34.2.36     58070      7680         6         Benign           0.887062    Brute Force           0.045634 Reconnaissance           0.024400
          3    10.5.30.64      10.34.2.36     56241      7680         6         Benign           0.964235 Reconnaissance           0.011388           DDoS           0.010980
          4 40.101.69.226      10.34.2.36       443     62610         6            Bot           0.902680         Benign          

In [10]:
CONFIDENCE_THRESHOLD = 0.80

prediction_report["decision"] = np.where(
    prediction_report["top_1_probability"]
    >= CONFIDENCE_THRESHOLD,

    prediction_report["top_1_label"],

    "Needs Review"
)

print(
    prediction_report[
        [
            "flow_index",
            "src_ip",
            "dst_ip",
            "src_port",
            "dst_port",
            "top_1_label",
            "top_1_probability",
            "decision"
        ]
    ].to_string(index=False)
)

 flow_index     src_ip          dst_ip  src_port  dst_port top_1_label  top_1_probability     decision
          0  10.0.56.1       224.0.0.5         0         0        MITM           0.736116 Needs Review
          1  10.0.29.2       224.0.0.5         0         0        MITM           0.728288 Needs Review
          2  10.0.30.1       224.0.0.5         0         0        MITM           0.736116 Needs Review
          3  10.0.29.1       224.0.0.5         0         0        MITM           0.736212 Needs Review
          4  10.0.30.2       224.0.0.5         0         0        MITM           0.737717 Needs Review
          5  10.0.56.2       224.0.0.5         0         0        MITM           0.736007 Needs Review
          6  10.0.67.1       224.0.0.5         0         0      Benign           0.869781       Benign
          7 10.0.67.23     224.0.0.251     12345      5353      Benign           0.473189 Needs Review
          8 10.0.67.23     224.0.0.251     35454      5353      Benign   

In [11]:
# ============================================================
# PROBLEMLI FEATURE TESPIT HUcRESI
# ============================================================

from pathlib import Path
import joblib
import numpy as np
import pandas as pd


ARTIFACT_DIR = Path("artifacts")

MODEL_FEATURES_PATH = (
    ARTIFACT_DIR / "feature_names.joblib"
)

GENERATED_CSV_PATH = Path(
    "deneme_nfv2_features.csv"
)

# Eğitimde kullanılan ana CSV'nin yolu
TRAIN_CSV_PATH = Path(
    "data/NF-UQ-NIDS-v2.csv"
)


# ------------------------------------------------------------
# 1. Modelin beklediği feature isimleri
# ------------------------------------------------------------

model_features = list(
    joblib.load(MODEL_FEATURES_PATH)
)

print("Modelin bekledigi feature sayisi:")
print(len(model_features))


# ------------------------------------------------------------
# 2. Converter çıktısını yükle
# ------------------------------------------------------------

if "features_df" in globals():
    generated_df = features_df.copy()

    print(
        "\nConverter verisi features_df "
        "degiskeninden alindi."
    )

elif GENERATED_CSV_PATH.exists():
    generated_df = pd.read_csv(
        GENERATED_CSV_PATH,
        low_memory=False,
    )

    print(
        "\nConverter verisi CSV'den alindi:"
    )
    print(GENERATED_CSV_PATH.resolve())

else:
    raise FileNotFoundError(
        "Ne features_df bulundu ne de "
        f"{GENERATED_CSV_PATH} bulundu."
    )


# ------------------------------------------------------------
# 3. Eğitim datasını yükle
# ------------------------------------------------------------

if not TRAIN_CSV_PATH.exists():
    raise FileNotFoundError(
        "Egitim CSV'si bulunamadi:\n"
        f"{TRAIN_CSV_PATH.resolve()}"
    )

# ESKI KODU SIL:
# training_df = pd.read_csv(
#     TRAIN_CSV_PATH,
#     low_memory=False,
# )

# YENI KOD:
TRAIN_ROW_CAP = 200_000
TRAIN_CHUNK_SIZE = 50_000

training_parts = []
total_rows = 0

for chunk in pd.read_csv(
    TRAIN_CSV_PATH,
    chunksize=TRAIN_CHUNK_SIZE,
    low_memory=True,
    on_bad_lines="skip",
    engine="c"
):
    remaining = TRAIN_ROW_CAP - total_rows

    if remaining <= 0:
        break

    if len(chunk) > remaining:
        chunk = chunk.iloc[:remaining].copy()

    # RAM kullanımını azalt
    float_columns = chunk.select_dtypes(
        include=["float64"]
    ).columns

    integer_columns = chunk.select_dtypes(
        include=["int64"]
    ).columns

    chunk[float_columns] = chunk[
        float_columns
    ].astype(np.float32)

    for column in integer_columns:
        chunk[column] = pd.to_numeric(
            chunk[column],
            downcast="integer"
        )

    training_parts.append(chunk)

    total_rows += len(chunk)

    print(
        f"\rOkunan satir: {total_rows:,}",
        end=""
    )

training_df = pd.concat(
    training_parts,
    ignore_index=True
)

del training_parts

print("\n")
print("Training shape:", training_df.shape)

print(
    "Training bellek:",
    round(
        training_df.memory_usage(
            deep=True
        ).sum() / 1024**2,
        2
    ),
    "MB"
)


# ------------------------------------------------------------
# 4. Eksik ve fazla kolonlar
# ------------------------------------------------------------

missing_in_generated = [
    feature
    for feature in model_features
    if feature not in generated_df.columns
]

extra_in_generated = [
    column
    for column in generated_df.columns
    if column not in model_features
]

print("\nConverter'da eksik feature'lar:")
print(missing_in_generated)

print("\nConverter'daki fazla kolonlar:")
print(extra_in_generated)


# ------------------------------------------------------------
# 5. Sadece ortak model feature'larını sayısala çevir
# ------------------------------------------------------------

common_features = [
    feature
    for feature in model_features
    if (
        feature in generated_df.columns
        and feature in training_df.columns
    )
]

generated_numeric = generated_df[
    common_features
].copy()

training_numeric = training_df[
    common_features
].copy()

for feature in common_features:
    generated_numeric[feature] = pd.to_numeric(
        generated_numeric[feature],
        errors="coerce",
    )

    training_numeric[feature] = pd.to_numeric(
        training_numeric[feature],
        errors="coerce",
    )

generated_numeric = generated_numeric.replace(
    [np.inf, -np.inf],
    np.nan,
)

training_numeric = training_numeric.replace(
    [np.inf, -np.inf],
    np.nan,
)


# ------------------------------------------------------------
# 6. Her feature için kalite ve dağılım kontrolü
# ------------------------------------------------------------

report_rows = []

for feature in model_features:
    row = {
        "feature": feature,
        "exists_in_generated":
            feature in generated_df.columns,
        "exists_in_training":
            feature in training_df.columns,
        "problems": [],
    }

    if feature not in generated_df.columns:
        row["problems"].append(
            "CONVERTERDA_EKSIK"
        )

        report_rows.append(row)
        continue

    if feature not in training_df.columns:
        row["problems"].append(
            "TRAINING_DATASINDA_EKSIK"
        )

        report_rows.append(row)
        continue

    generated_series = generated_numeric[feature]
    training_series = training_numeric[feature]

    generated_valid = generated_series.dropna()
    training_valid = training_series.dropna()

    row["generated_nan_ratio"] = float(
        generated_series.isna().mean()
    )

    row["training_nan_ratio"] = float(
        training_series.isna().mean()
    )

    row["generated_zero_ratio"] = float(
        (generated_valid == 0).mean()
    ) if len(generated_valid) else 1.0

    row["training_zero_ratio"] = float(
        (training_valid == 0).mean()
    ) if len(training_valid) else 1.0

    row["generated_unique"] = int(
        generated_valid.nunique()
    )

    row["training_unique"] = int(
        training_valid.nunique()
    )

    row["generated_min"] = (
        float(generated_valid.min())
        if len(generated_valid)
        else np.nan
    )

    row["generated_median"] = (
        float(generated_valid.median())
        if len(generated_valid)
        else np.nan
    )

    row["generated_max"] = (
        float(generated_valid.max())
        if len(generated_valid)
        else np.nan
    )

    row["training_min"] = (
        float(training_valid.min())
        if len(training_valid)
        else np.nan
    )

    row["training_median"] = (
        float(training_valid.median())
        if len(training_valid)
        else np.nan
    )

    row["training_max"] = (
        float(training_valid.max())
        if len(training_valid)
        else np.nan
    )

    # Converter tamamen NaN üretmiş
    if generated_series.isna().all():
        row["problems"].append(
            "TAMAMI_NAN"
        )

    # Converter yalnızca tek değer üretmiş
    if generated_valid.nunique() <= 1:
        row["problems"].append(
            "SABIT_DEGER"
        )

    # Converter hep sıfır üretiyor fakat training'de sıfır değil
    if (
        row["generated_zero_ratio"] >= 0.99
        and row["training_zero_ratio"] < 0.99
    ):
        row["problems"].append(
            "CONVERTER_NEREDEYSE_HEP_SIFIR"
        )

    # Üretilen değer training aralığının tamamen dışında
    if (
        len(generated_valid)
        and len(training_valid)
        and (
            generated_valid.min()
            > training_valid.max()
            or generated_valid.max()
            < training_valid.min()
        )
    ):
        row["problems"].append(
            "TRAINING_ARALIGIYLA_KESISMİYOR"
        )

    # Medyan ölçeği çok farklı
    generated_abs_median = abs(
        row["generated_median"]
    )

    training_abs_median = abs(
        row["training_median"]
    )

    if (
        generated_abs_median > 0
        and training_abs_median > 0
    ):
        scale_ratio = max(
            generated_abs_median,
            training_abs_median,
        ) / min(
            generated_abs_median,
            training_abs_median,
        )

        row["median_scale_ratio"] = float(
            scale_ratio
        )

        if scale_ratio >= 100:
            row["problems"].append(
                "MEDYAN_OLCEGI_100X_FARKLI"
            )
    else:
        row["median_scale_ratio"] = np.nan

    report_rows.append(row)


# ------------------------------------------------------------
# 7. Raporu oluştur
# ------------------------------------------------------------

feature_audit_df = pd.DataFrame(
    report_rows
)

feature_audit_df["problem_count"] = (
    feature_audit_df["problems"]
    .apply(len)
)

feature_audit_df["problems"] = (
    feature_audit_df["problems"]
    .apply(
        lambda values: ", ".join(values)
    )
)

problematic_features_df = (
    feature_audit_df[
        feature_audit_df["problem_count"] > 0
    ]
    .sort_values(
        [
            "problem_count",
            "feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 8. Sonuçları yazdır ve kaydet
# ------------------------------------------------------------

print("\n========================================")
print("PROBLEMLI FEATURE SAYISI")
print("========================================")
print(len(problematic_features_df))

print("\n========================================")
print("PROBLEMLI FEATURE LISTESI")
print("========================================")

if problematic_features_df.empty:
    print(
        "Otomatik kontrolde problem bulunmadi."
    )
else:
    for _, row in problematic_features_df.iterrows():
        print(
            f"- {row['feature']}: "
            f"{row['problems']}"
        )

print("\n========================================")
print("DETAYLI RAPOR")
print("========================================")

display(
    problematic_features_df[
        [
            "feature",
            "problems",
            "generated_zero_ratio",
            "training_zero_ratio",
            "generated_unique",
            "training_unique",
            "generated_min",
            "generated_median",
            "generated_max",
            "training_min",
            "training_median",
            "training_max",
            "median_scale_ratio",
        ]
    ]
)

feature_audit_df.to_csv(
    "feature_audit_all.csv",
    index=False,
)

problematic_features_df.to_csv(
    "problematic_features.csv",
    index=False,
)

print("\nKaydedildi:")
print(
    Path("feature_audit_all.csv").resolve()
)
print(
    Path("problematic_features.csv").resolve()
)

Modelin bekledigi feature sayisi:
39

Converter verisi CSV'den alindi:
C:\Users\egemen.keles\PycharmProjects\mitre_mapper\deneme_nfv2_features.csv
Okunan satir: 200,000

Training shape: (200000, 46)
Training bellek: 73.09 MB

Converter'da eksik feature'lar:
[]

Converter'daki fazla kolonlar:
[]

PROBLEMLI FEATURE SAYISI
20

PROBLEMLI FEATURE LISTESI
- CLIENT_TCP_FLAGS: SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR
- DNS_TTL_ANSWER: SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR
- DST_TO_SRC_AVG_THROUGHPUT: SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR
- DST_TO_SRC_SECOND_BYTES: SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR
- DURATION_OUT: SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR
- NUM_PKTS_1024_TO_1514_BYTES: SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR
- OUT_BYTES: SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR
- OUT_PKTS: SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR
- RETRANSMITTED_IN_BYTES: SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR
- RETRANSMITTED_IN_PKTS: SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR
-

,feature,problems,generated_zero_ratio,training_zero_ratio,generated_unique,training_unique,generated_min,generated_median,generated_max,training_min,training_median,training_max,median_scale_ratio
0,CLIENT_TCP_FLAGS,"SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR",1.000000,0.384810,1,33,0.0,0.0,0.0,0.0,2.0,2.230000e+02,NaN
1,DNS_TTL_ANSWER,"SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR",1.000000,0.881660,1,1213,0.0,0.0,0.0,0.0,0.0,4.294916e+09,NaN
2,DST_TO_SRC_AVG_THROUGHPUT,"SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR",1.000000,0.543215,1,5472,0.0,0.0,0.0,0.0,0.0,4.281120e+09,NaN
3,DST_TO_SRC_SECOND_BYTES,"SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR",1.000000,0.537063,1,6994,0.0,0.0,0.0,0.0,0.0,4.805721e+22,NaN
4,DURATION_OUT,"SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR",1.000000,0.931415,1,789,0.0,0.0,0.0,0.0,0.0,3.854700e+04,NaN
5,NUM_PKTS_1024_TO_1514_BYTES,"SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR",1.000000,0.891730,1,261,0.0,0.0,0.0,0.0,0.0,8.228900e+04,NaN
6,OUT_BYTES,"SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR",1.000000,0.537060,1,6317,0.0,0.0,0.0,0.0,0.0,1.235906e+08,NaN
7,OUT_PKTS,"SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR",1.000000,0.537060,1,326,0.0,0.0,0.0,0.0,0.0,8.269400e+04,NaN
8,RETRANSMITTED_IN_BYTES,"SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR",1.000000,0.889940,1,972,0.0,0.0,0.0,0.0,0.0,5.313040e+05,NaN
9,RETRANSMITTED_IN_PKTS,"SABIT_DEGER, CONVERTER_NEREDEYSE_HEP_SIFIR",1.000000,0.889940,1,71,0.0,0.0,0.0,0.0,0.0,3.980000e+02,NaN



Kaydedildi:
C:\Users\egemen.keles\PycharmProjects\mitre_mapper\feature_audit_all.csv
C:\Users\egemen.keles\PycharmProjects\mitre_mapper\problematic_features.csv


In [15]:
print(
    features["FLOW_DURATION_MILLISECONDS"]
    .describe(
        percentiles=[
            .25,.5,.75,.9,.95,.99
        ]
    )
)

count      6.000000
mean      75.374084
std      158.084290
min        0.000000
25%        0.783861
50%        7.087588
75%       33.969940
90%      219.034668
95%      307.745331
99%      378.713861
max      396.455994
Name: FLOW_DURATION_MILLISECONDS, dtype: float64


In [17]:
print(
    training_df["FLOW_DURATION_MILLISECONDS"]
    .describe(
        percentiles=[
            .25,.5,.75,.9,.95,.99
        ]
    )
)

count    2.000000e+05
mean     2.328964e+06
std      2.139265e+06
min      0.000000e+00
25%      0.000000e+00
50%      4.293077e+06
75%      4.294029e+06
90%      4.294561e+06
95%      4.294905e+06
99%      4.294952e+06
max      4.294966e+06
Name: FLOW_DURATION_MILLISECONDS, dtype: float64


In [12]:
for flow_key, flow in flows.items():
    if flow_key[4] == 89:
        print(
            "FLOW:", flow_key,
            "IN_PKTS:", flow["IN_PKTS"],
            "OUT_PKTS:", flow["OUT_PKTS"],
            "DURATION_MS:",
            round(
                (flow["last_ts"] - flow["first_ts"]) * 1000,
                3
            )
        )

FLOW: ('10.0.56.1', '224.0.0.5', 0, 0, 89) IN_PKTS: 1627 OUT_PKTS: 0 DURATION_MS: 3182588.247
FLOW: ('10.0.29.2', '224.0.0.5', 0, 0, 89) IN_PKTS: 1611 OUT_PKTS: 0 DURATION_MS: 3182225.439
FLOW: ('10.0.30.1', '224.0.0.5', 0, 0, 89) IN_PKTS: 1622 OUT_PKTS: 0 DURATION_MS: 3182196.659
FLOW: ('10.0.29.1', '224.0.0.5', 0, 0, 89) IN_PKTS: 1620 OUT_PKTS: 0 DURATION_MS: 3182196.634
FLOW: ('10.0.30.2', '224.0.0.5', 0, 0, 89) IN_PKTS: 1614 OUT_PKTS: 0 DURATION_MS: 3182196.588
FLOW: ('10.0.56.2', '224.0.0.5', 0, 0, 89) IN_PKTS: 1625 OUT_PKTS: 0 DURATION_MS: 3182196.572
FLOW: ('10.0.67.1', '224.0.0.5', 0, 0, 89) IN_PKTS: 1588 OUT_PKTS: 0 DURATION_MS: 3182196.562


In [1]:
!pip install pyshark scapy pandas numpy joblib

   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ---------------------------------------- 4.1/4.1 MB 157.0 MB/s  0:00:00

   -------------------- ------------------- 2/4 [lxml]
   -------------------- ------------------- 2/4 [lxml]
   ---------------------------------------- 4/4 [pyshark]




[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path

TSHARK_PATH = Path(r"C:\Program Files\Wireshark\tshark.exe")
DUMPCAP_PATH = Path(r"C:\Program Files\Wireshark\dumpcap.exe")

print("TShark:", TSHARK_PATH.exists())
print("Dumpcap:", DUMPCAP_PATH.exists())

TShark: True
Dumpcap: True
